## Prompt Evaluation

_Setup_

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

claude_api_key = os.getenv("CLAUDE_API_KEY")

# client = Anthropic(base_url="http://localhost:11434", api_key="ollama")
client = Anthropic(api_key=claude_api_key)

# model = "gemma4:e4b-mlx"
model = "claude-sonnet-4-5"
max_tokens = 1000

_Utility Functions_

In [2]:
def add_user_message(messages, message):
  messages.append({"role": "user", "content": message})

def add_assistant_message(messages, message):
  messages.append({"role": "assistant", "content": message})

def chat(messages, **kargs):
  params = {
    "model": model,
    "max_tokens": max_tokens,
    "messages": messages,
    **kargs
  }

  response = client.messages.create(**params)

  answer = ""
  for block in response.content:
    if (block.type == "text"):
      answer = block.text

  return answer

In [3]:
import json
def print_json(_a):
  print(json.dumps(_a, indent=2))

In [4]:
def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "output format: 'python' or 'json' or 'regex'",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [5]:
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
import ast
import json
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [7]:
# Merge our prompt with user's task / test-case and send it to claude to get output
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

<task>
{test_case["task"]}
</task>

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

# Create evaluation prompt
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    The task against which the prompt needs to be evaluated
    <task>
    {test_case["task"]}
    </task>
    
    Solution:
    <solution>
    {output}
    </solution>

    Key criteria for the acceptable solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def grade_syntax(test_case, output):
    if (test_case['format'] == 'python'): return validate_python(output)
    if (test_case['format'] == 'json'): return validate_json(output)
    if (test_case['format'] == 'regex'): return validate_regex(output)
    return 0

# Calls run_prompt, then grades the result
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grading
    model_grade = grade_by_model(test_case, output)
    reasoning = model_grade["reasoning"]
    model_score = model_grade["score"]
    syntax_score = grade_syntax(test_case, output)
    score = (model_score + syntax_score) / 2
    
    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning
    }

# Loads the dataset and calls run_test_case with each case
def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:

from statistics import mean

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
average_score = mean([result["score"] for result in results])
print(f"Average score: {average_score}")

In [ ]:
print_json(results)